# Subword Seq2Seq — No-Preprocessing Baseline (trains directly on raw train.csv)

**How to switch modes:** set `EXECUTION_MODE` in the next cell to `"online"` or `"offline"`.

- `"online"` — installs packages, logs in to Hugging Face, tracks the run in MLflow/DagsHub, and pushes the final model/tokenizer to the Hub. Requires Kaggle's internet toggle **on**.
- `"offline"` — skips all of the above, loads the base model from a local Kaggle Dataset input instead of downloading it, and saves the trained model/tokenizer to the working directory instead of pushing to the Hub. Required for the actual competition submission run, since Kaggle disables internet on the scored run.

In [10]:
EXECUTION_MODE = "offline"  # "online" or "offline" — flip this before a real (scored) submission run
assert EXECUTION_MODE in ("online", "offline")
IS_ONLINE = EXECUTION_MODE == "online"
print(f"Running in {EXECUTION_MODE.upper()} mode")


Running in OFFLINE mode


In [11]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [12]:
import sys
import subprocess

if IS_ONLINE:
    # Package installs require internet — skipped entirely in offline mode.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dacite>=1.9,<2", "--upgrade"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "transformers", "datasets", "huggingface_hub", "accelerate", "mlflow"], check=True)
    # Force dagshub to skip checking dependencies
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dagshub", "--no-deps"], check=True)
else:
    print("Offline mode: skipping package installation (using the environment's pre-installed packages).")


Offline mode: skipping package installation (using the environment's pre-installed packages).


In [13]:
from transformers import set_seed 

# Set global random seed for reproducibility
set_seed(42)

In [14]:
if IS_ONLINE:
    import mlflow
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login

    user_secrets = UserSecretsClient()

    # Authorize Hugging Face
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)

    # Pass Dagshub credentials from Kaggle Secrets to Environment Variables
    os.environ["MLFLOW_TRACKING_USERNAME"] = user_secrets.get_secret("MLFLOW_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = user_secrets.get_secret("MLFLOW_PASSWORD")
    MLFLOW_TRACKING_URI = user_secrets.get_secret("MLFLOW_TRACKING_URI")

    # Connect to MLflow server and select the experiment
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment("deep-past-initiative-machine-translation")

    # Check connection with a test run
    with mlflow.start_run(run_name="New Data-Preprocessing Pipeline Test"):
        mlflow.set_tag("author", "author")
        mlflow.log_param("architecture", "mBART / mT5 / Subword Seq2Seq")
        mlflow.log_metric("status", 1.0)
        print("Successful MLflow-Dagshub connection!")
else:
    print("Offline mode: skipping Hugging Face login and MLflow/DagsHub tracking.")


Offline mode: skipping Hugging Face login and MLflow/DagsHub tracking.


## Data — raw `train.csv`, untouched

No normalization, no sentence-boundary extraction, no anchor matching. The only operation applied is `dropna` on the two text columns (a row with a missing source or target can't be trained on at all — this isn't preprocessing of the text itself, just discarding unusable rows) plus a `split` column, since `train.csv` doesn't ship with one.

The split is **document-level** using the same seeded random assignment approach as the pipeline notebook, so a document's style/vocabulary can't leak between train and val — this is a split decision, not a transformation of the data itself, so it's kept even in this "no preprocessing" baseline.

In [15]:
import random
import pandas as pd
from datasets import Dataset, DatasetDict

RAW_TRAIN_PATH = "/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv"
VAL_FRACTION = 0.1

df = pd.read_csv(RAW_TRAIN_PATH)
df = df.dropna(subset=["transliteration", "translation"]).reset_index(drop=True)

# Pick whatever document-id-like column is present; fall back to a plain row-level
# split (with a printed warning) if the raw file doesn't have one.
_id_col = "oare_id"

rng = random.Random(42)
if _id_col is not None:
    doc_ids = sorted(df[_id_col].dropna().unique().tolist())
    rng.shuffle(doc_ids)
    n_val_docs = max(1, int(len(doc_ids) * VAL_FRACTION))
    val_ids = set(doc_ids[:n_val_docs])
    df["split"] = df[_id_col].apply(lambda x: "val" if x in val_ids else "train")
else:
    print(f"WARNING: no document-id column found among {list(df.columns)} — "
          f"falling back to a row-level random split (train.csv is document-level, "
          f"so this risks a document's style leaking between train and val).")
    idx = list(df.index)
    rng.shuffle(idx)
    n_val = max(1, int(len(idx) * VAL_FRACTION))
    val_idx = set(idx[:n_val])
    df["split"] = ["val" if i in val_idx else "train" for i in df.index]

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
})

print(f"train: {len(train_df)} rows, val: {len(val_df)} rows "
      f"({'by document' if _id_col else 'by row (fallback)'})")


train: 1405 rows, val: 156 rows (by document)


In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"
# Local/attached-dataset path used in offline mode (no internet download).
MODEL_PATH = "/kaggle/input/models/nensipansuriya1311/facebookmbart-large-50-many-to-many-mmt/pytorch/default/1"

MODEL_SOURCE = MODEL_NAME if IS_ONLINE else MODEL_PATH

tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE)

tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "en_XX"

MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256

def preprocess_function(examples):
    inputs = examples["transliteration"]
    targets = examples["translation"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_SOURCE)


Map:   0%|          | 0/1405 [00:00<?, ? examples/s]

Map:   0%|          | 0/156 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

## Metrics — manual implementation

Pure-Python BLEU / chrF++, no `evaluate`/`sacrebleu` dependency. This avoids a metric-script download from the Hugging Face Hub, which `evaluate.load(...)` would otherwise require even in "online" mode.

In [17]:
import math
from collections import Counter
import numpy as np

def _ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

def corpus_bleu(preds, refs, max_n=4):
    clipped_counts = [0] * max_n
    total_counts = [0] * max_n
    pred_len_total, ref_len_total = 0, 0
    for pred, ref in zip(preds, refs):
        p_tokens, r_tokens = pred.split(), ref.split()
        pred_len_total += len(p_tokens)
        ref_len_total += len(r_tokens)
        for n in range(1, max_n + 1):
            p_counts = _ngram_counts(p_tokens, n)
            r_counts = _ngram_counts(r_tokens, n)
            clipped_counts[n - 1] += sum(min(c, r_counts[g]) for g, c in p_counts.items())
            total_counts[n - 1] += max(sum(p_counts.values()), 0)
    precisions = [clipped_counts[n] / total_counts[n] if total_counts[n] > 0 else 0.0 for n in range(max_n)]
    geo_mean = 0.0 if min(precisions) == 0 else math.exp(sum(math.log(p) for p in precisions) / max_n)
    bp = 1.0 if pred_len_total > ref_len_total else math.exp(1 - ref_len_total / max(pred_len_total, 1))
    return 100.0 * bp * geo_mean

def corpus_chrf(preds, refs, n=6, beta=2):
    tp = [0] * n; tt = [0] * n; rc = [0] * n; rt = [0] * n
    for pred, ref in zip(preds, refs):
        p_chars, r_chars = pred.replace(" ", ""), ref.replace(" ", "")
        for k in range(1, n + 1):
            p_counts, r_counts = _ngram_counts(p_chars, k), _ngram_counts(r_chars, k)
            overlap = sum((p_counts & r_counts).values())
            tp[k - 1] += overlap; tt[k - 1] += max(sum(p_counts.values()), 0)
            rc[k - 1] += overlap; rt[k - 1] += max(sum(r_counts.values()), 0)
    precisions = [tp[k] / tt[k] if tt[k] > 0 else 0.0 for k in range(n)]
    recalls = [rc[k] / rt[k] if rt[k] > 0 else 0.0 for k in range(n)]
    avg_p, avg_r = sum(precisions) / n, sum(recalls) / n
    if avg_p + avg_r == 0:
        return 0.0
    beta2 = beta ** 2
    return 100.0 * (1 + beta2) * avg_p * avg_r / (beta2 * avg_p + avg_r)

def geometric_mean(a, b):
    """Geometric mean of two non-negative metrics (chrF++ and BLEU, both on a 0-100 scale)."""
    a, b = max(a, 0.0), max(b, 0.0)
    return math.sqrt(a * b)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace -100 in labels so they decode correctly
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    chrf_pp = corpus_chrf(decoded_preds, decoded_labels)  # beta=2 approximates chrF++
    bleu = corpus_bleu(decoded_preds, decoded_labels)
    geo_mean = geometric_mean(chrf_pp, bleu)

    return {
        "chrf_pp": round(chrf_pp, 4),
        "bleu": round(bleu, 4),
        "geo_mean_chrf_bleu": round(geo_mean, 4),
    }


## Training

Memory optimizations added on top of the original setup (all safe to use in both modes):

- **`group_by_length=True`** — batches similar-length examples together so padding waste (and the peak activation memory that comes with it) is minimized, instead of a batch occasionally pairing a short and a near-256-token example.
- **`eval_accumulation_steps=1`** — with `predict_with_generate=True`, the Trainer otherwise keeps *all* generated eval predictions resident on the GPU until the full eval pass finishes; this was the most likely source of eval-time OOM, independent of `MAX_TARGET_LENGTH`. Setting this moves each batch's predictions to CPU immediately.
- **Explicit `use_cache` toggling** — `use_cache=True` (the default, used by `.generate()`) is incompatible with `gradient_checkpointing=True` and silently wastes memory/emits warnings during training if left on. It's turned off before `trainer.train()` and back on before generation/prediction.

**Per-epoch MLflow logging** (online mode only, via the `MLflowEpochLogger` callback): `train_loss`, `val_loss`, `chrf_pp`, `bleu`, and `geo_mean_chrf_bleu` (the geometric mean of chrF++ and BLEU) are logged once per epoch, keyed by epoch number, so the run's metric charts in MLflow/DagsHub show all five curves across training. Offline mode skips this entirely — the callback list is empty and training/eval still run and print `eval_metrics` normally.

In [18]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, TrainerCallback
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

HF_REPO_NAME = user_secrets.get_secret("HF_REPO_NAME")
RUN_NAME = "mbart50-train-csv"

set_seed(42)

training_args = Seq2SeqTrainingArguments(
    output_dir="./results_mbart",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    optim="adafactor",
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=1,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="chrf_pp",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=False,
    group_by_length=True,          # minimize padding waste per batch
    eval_accumulation_steps=1,     # stream eval predictions to CPU instead of accumulating on GPU
)

model.gradient_checkpointing_enable()
model.config.use_cache = False  # required alongside gradient checkpointing during training

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


class MLflowEpochLogger(TrainerCallback):
    """
    Logs train loss, val loss, chrF++, BLEU, and their geometric mean to MLflow once
    per epoch (a no-op when IS_ONLINE is False or no MLflow run is active). Training
    loss is captured from the last regular `on_log` call (Trainer logs it every
    `logging_steps`) and paired up with that epoch's eval metrics in `on_evaluate`,
    since eval_strategy="epoch" fires `on_evaluate` once at the end of each epoch.
    """

    def __init__(self):
        self._last_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs and "eval_loss" not in logs:
            self._last_train_loss = logs["loss"]

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not IS_ONLINE or metrics is None or mlflow.active_run() is None:
            return

        epoch = int(round(state.epoch)) if state.epoch is not None else state.global_step
        eval_loss = metrics.get("eval_loss")
        chrf_pp = metrics.get("eval_chrf_pp")
        bleu = metrics.get("eval_bleu")
        geo_mean = metrics.get("eval_geo_mean_chrf_bleu")
        if geo_mean is None and chrf_pp is not None and bleu is not None:
            geo_mean = geometric_mean(chrf_pp, bleu)

        epoch_metrics = {}
        if self._last_train_loss is not None:
            epoch_metrics["train_loss"] = self._last_train_loss
        if eval_loss is not None:
            epoch_metrics["val_loss"] = eval_loss
        if chrf_pp is not None:
            epoch_metrics["chrf_pp"] = chrf_pp
        if bleu is not None:
            epoch_metrics["bleu"] = bleu
        if geo_mean is not None:
            epoch_metrics["geo_mean_chrf_bleu"] = geo_mean

        if epoch_metrics:
            mlflow.log_metrics(epoch_metrics, step=epoch)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[MLflowEpochLogger()] if IS_ONLINE else [],
)

if IS_ONLINE:
    with mlflow.start_run(run_name=RUN_NAME):
        mlflow.set_tag("author", "author")
        mlflow.set_tag("model_architecture", "mbart50")
        mlflow.set_tag("hf_repo", HF_REPO_NAME)
        mlflow.set_tag("preprocessing", "none (raw train.csv)")

        mlflow.log_params({
            "model_name": MODEL_SOURCE,
            "lr": training_args.learning_rate,
            "batch_size": training_args.per_device_train_batch_size,
            "epochs": training_args.num_train_epochs,
            "max_src_len": MAX_SOURCE_LENGTH,
            "max_tgt_len": MAX_TARGET_LENGTH,
            "data_columns": "raw_train_csv_untouched"
        })

        trainer.train()

        eval_metrics = trainer.evaluate()
        mlflow.log_metrics({
            "val_chrF_pp": eval_metrics["eval_chrf_pp"],
            "val_bleu": eval_metrics["eval_bleu"],
            "val_loss": eval_metrics["eval_loss"],
            "val_geo_mean_chrf_bleu": eval_metrics["eval_geo_mean_chrf_bleu"],
        })
else:
    trainer.train()
    eval_metrics = trainer.evaluate()
    print(eval_metrics)

# Re-enable the KV cache now that training/eval (which used gradient checkpointing) is done —
# needed for efficient generation below.
model.config.use_cache = True


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
if IS_ONLINE:
    # Publish the trained model to the Hugging Face Hub
    model.push_to_hub('subword-seq2seq-model-raw', private=True)
    tokenizer.push_to_hub("subword-seq2seq-tokenizer-raw", private=True)
else:
    # No internet available — save locally instead. These land in Kaggle's working directory
    # and can be kept as notebook output / attached to a new dataset version if you want to reuse the weights.
    model.save_pretrained("./subword-seq2seq-model-raw")
    tokenizer.save_pretrained("./subword-seq2seq-tokenizer-raw")
    print("Saved model and tokenizer locally (offline mode).")


In [ ]:
# Generate predictions on the validation dataset
raw_predictions = trainer.predict(tokenized_datasets["validation"])
preds = np.where(raw_predictions.predictions != -100, raw_predictions.predictions, tokenizer.pad_token_id)
decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

_val_id_col = "oare_id"
val_ids = val_df[_val_id_col].values if _val_id_col else val_df.index.values

val_df_results = pd.DataFrame({
    "id": val_ids,
    "source_text": val_df["transliteration"].values,
    "target_text": val_df["translation"].values,
    "pred_translation": [p.strip() for p in decoded_preds]
})

val_df_results.to_csv("preds_val_mbart50_raw.csv", index=False)
print("Predictions file preds_val_mbart50_raw.csv saved successfully!")


## Test-set inference (submission.csv)

Replaces the original row-by-row generation loop with **length-sorted, dynamically-padded mini-batches**:

- Sorting by tokenized length before batching means each batch pads only to the length of its *longest member*, not to a fixed worst case — this cuts wasted compute/memory versus batching in file order.
- Batches (not single examples) get better GPU utilization than the original loop, without the memory risk of one big fixed-size batch, since batches are still capped at `GEN_BATCH_SIZE` and periodically cleared.
- If a batch still OOMs (e.g. a batch of unusually long examples), the batch size is halved and retried automatically instead of crashing the whole run.
- Results are reordered back to the original file order before writing `submission.csv`.

In [ ]:
import os
import gc
import pandas as pd
import torch
from tqdm import tqdm

test_path = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
test_df = pd.read_csv(test_path)

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

forced_bos_token_id = tokenizer.lang_code_to_id.get("en_XX")

GEN_BATCH_SIZE = 16  # starting batch size

texts = test_df["transliteration"].tolist()
# Sort by tokenized length so each batch pads to its own max, not a global worst case
order = sorted(range(len(texts)), key=lambda i: len(tokenizer(texts[i], truncation=True, max_length=MAX_SOURCE_LENGTH)["input_ids"]))
sorted_texts = [texts[i] for i in order]

predictions_sorted = [None] * len(sorted_texts)

def generate_batch(batch_texts):
    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        padding=True,
    ).to(device)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            pad_token_id=tokenizer.pad_token_id,
            forced_bos_token_id=forced_bos_token_id,
        )
    return [tokenizer.decode(o, skip_special_tokens=True).strip() for o in outputs]

pbar = tqdm(total=len(sorted_texts), desc="Generating Submissions")
i = 0
batch_size = GEN_BATCH_SIZE
while i < len(sorted_texts):
    chunk = sorted_texts[i:i + batch_size]
    try:
        decoded = generate_batch(chunk)
        for j, text in enumerate(decoded):
            predictions_sorted[i + j] = text
        i += len(chunk)
        pbar.update(len(chunk))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()
        if batch_size == 1:
            raise
        batch_size = max(1, batch_size // 2)
        print(f"CUDA OOM — reducing generation batch size to {batch_size} and retrying.")
    finally:
        if device == "cuda" and i % (GEN_BATCH_SIZE * 4) == 0:
            torch.cuda.empty_cache()
pbar.close()

predictions = [None] * len(texts)
for sorted_idx, original_idx in enumerate(order):
    predictions[original_idx] = predictions_sorted[sorted_idx]

submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": predictions
})

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv successfully!")
